# Session 9. Multi-agent systems and the research agent

**One loop becomes a team. Shared state is where the team meets.**

- first the plumbing: parallel branches, reducers, `Send`, subgraphs
- then one research agent end to end: analysts, interviews, one merged report
- demo doubles as practice today: you rebuild it over your own corpus

In [ ]:
import operator
import os
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()  # reads .env once; nothing below opens a file


def chat_model(size: str = "cheap", **kwargs):
    """A model object for the configured provider. A dozen lines, copy them once."""
    name = os.environ[f"MODEL_{size.upper()}"]  # ids live in .env, never in code
    secret = os.environ["LLM_API_KEY"]
    if os.getenv("LLM_REASONING_EFFORT"):  # gpt-5.x: tools need reasoning "none"
        kwargs.setdefault("reasoning_effort", os.environ["LLM_REASONING_EFFORT"])
    # Gemini over OpenAI-compat drops the reasoning signature: turn two 400s
    if os.getenv("LLM_PROVIDER", "openai_compat") == "google_genai":
        return init_chat_model(f"google_genai:{name}", api_key=secret, **kwargs)
    return init_chat_model(
        f"openai:{name}", api_key=secret, base_url=os.environ["LLM_BASE_URL"], **kwargs
    )


print("provider:", os.getenv("LLM_PROVIDER", "openai_compat"),
      "| strong:", os.environ["MODEL_STRONG"])

## Parallel branches and write conflicts

**Two edges out of one node. Both destinations run in the same superstep.**

- two nodes write `notes` at once; the default policy is replacement
- two replacements in one step is not a race, it is an error
- the graph refuses loudly at run time: read the message

In [ ]:
from langgraph.errors import InvalidUpdateError
from langgraph.graph import END, START, StateGraph


class Clash(TypedDict):
    topic: str
    notes: list  # no reducer: a write replaces, and two at once collide


def ecology_note(state: Clash) -> dict:
    return {"notes": ["Pollination improves within two miles of a hive."]}


def rules_note(state: Clash) -> dict:
    return {"notes": ["Most cities cap the number of hives per roof."]}


builder = StateGraph(Clash)
builder.add_node("ecology", ecology_note)
builder.add_node("rules", rules_note)
builder.add_edge(START, "ecology")  # two edges out of START: both run this superstep
builder.add_edge(START, "rules")
builder.add_edge("ecology", END)
builder.add_edge("rules", END)

try:
    builder.compile().invoke({"topic": "rooftop hives", "notes": []},
                             config={"recursion_limit": 5})
except InvalidUpdateError as error:
    print("InvalidUpdateError:", error)

**A reducer is a merge policy. `operator.add` says: concatenate.**

- session 2 gave `messages` a reducer; every parallel key needs one
- merge order is task order: deterministic, run after run
- the node functions did not change, only one schema line

In [ ]:
class Survey(TypedDict):
    topic: str
    notes: Annotated[list, operator.add]  # a merge policy: concatenate the writes


builder = StateGraph(Survey)
builder.add_node("ecology", ecology_note)  # both node functions, untouched
builder.add_node("rules", rules_note)
builder.add_edge(START, "ecology")
builder.add_edge(START, "rules")
builder.add_edge("ecology", END)
builder.add_edge("rules", END)

merged = builder.compile().invoke({"topic": "rooftop hives", "notes": []},
                                  config={"recursion_limit": 5})
for note in merged["notes"]:  # merge order is task order: same every run
    print("-", note)

## Send and dynamic fan-out

**Static edges fan out a fixed number. `Send` reads the data first.**

- a conditional edge may return `Send` objects instead of a node name
- each `Send` carries a payload dict: that dict is the branch's whole state
- three questions today, five tomorrow: the branch count follows the data

In [ ]:
from langgraph.types import Send


class MapState(TypedDict):
    questions: list[str]
    answers: Annotated[list[str], operator.add]


class BranchState(TypedDict):
    question: str  # the Send payload is this branch's whole state


def plan(state: MapState) -> dict:
    return {}  # the fan-out happens on the edge below, not in a node


def fan_out(state: MapState) -> list[Send]:
    # one Send per question: the data picked the branch count
    return [Send("worker", {"question": q}) for q in state["questions"]]


def worker(state: BranchState) -> dict:
    return {"answers": [f"looked into {state['question']!r}"]}


builder = StateGraph(MapState)
builder.add_node("plan", plan)
builder.add_node("worker", worker)
builder.add_edge(START, "plan")
builder.add_conditional_edges("plan", fan_out, ["worker"])
builder.add_edge("worker", END)
mapper = builder.compile()

print(mapper.get_graph().draw_mermaid())

out = mapper.invoke(
    {"questions": ["hive density", "roof load", "sting liability"], "answers": []},
    config={"recursion_limit": 10},
)
print(out["answers"])

## Subgraphs

**A compiled graph is a node. Its state is its own business.**

- the child keeps private keys the parent schema never mentions
- one overlapping reduced key is the entire contract between them
- context isolation for free: branch scratch cannot leak into the report

In [ ]:
class ScoutState(TypedDict):
    district: str
    scratch: str  # private: the parent schema never mentions it
    sections: Annotated[list[str], operator.add]  # the one shared key


def survey_roofs(state: ScoutState) -> dict:
    return {"scratch": f"walked the {state['district']} rooftops at noon"}


def summarize(state: ScoutState) -> dict:
    return {"sections": [f"{state['district']}: {state['scratch']}"]}


child = StateGraph(ScoutState)
child.add_node("survey_roofs", survey_roofs)
child.add_node("summarize", summarize)
child.add_edge(START, "survey_roofs")
child.add_edge("survey_roofs", "summarize")
child.add_edge("summarize", END)


class CityState(TypedDict):
    district: str
    sections: Annotated[list[str], operator.add]


parent = StateGraph(CityState)
parent.add_node("scout", child.compile())  # a compiled graph, added as a node
parent.add_edge(START, "scout")
parent.add_edge("scout", END)

city = parent.compile().invoke({"district": "old town", "sections": []},
                               config={"recursion_limit": 5})
print(sorted(city))  # scratch never surfaced
print(city["sections"])

## Five orchestration patterns (Anthropic, Building effective agents)

- **prompt chaining**: fixed steps feed forward; use when the order is known upfront
- **routing**: classify, then dispatch to a specialist; use when inputs cluster into kinds
- **parallelization**: sectioning or voting; use when the parts are independent
- **orchestrator-workers**: a planner spawns workers per subtask it finds in the data
- **evaluator-optimizer**: generate, critique, retry; use when checking is cheaper than writing

**Routing you have met: session 2 parked a classifier, read-only.**

- the last session-2 cell classified text and routed nowhere
- today the same shape finally has alternatives worth routing to
- the classifier is a rule below; in production, often one cheap model call

In [ ]:
class Desk(TypedDict):
    question: str
    kind: str  # written by classify, read by the router
    reply: str


def classify(state: Desk) -> dict:
    q = state["question"].lower()
    if "permit" in q or "legal" in q:
        return {"kind": "rules"}
    if "sting" in q or "safe" in q:
        return {"kind": "safety"}
    return {"kind": "ecology"}


def route(state: Desk) -> str:
    return state["kind"]  # session 2's should_continue, with richer answers


def rules_desk(state: Desk) -> dict:
    return {"reply": "Check the city cap on hives per roof, then register."}


def safety_desk(state: Desk) -> dict:
    return {"reply": "Point flight paths away from terraces; stings are rare."}


def ecology_desk(state: Desk) -> dict:
    return {"reply": "Forage radius is about two miles; balconies bloom more."}


builder = StateGraph(Desk)
builder.add_node("classify", classify)
builder.add_node("rules", rules_desk)
builder.add_node("safety", safety_desk)
builder.add_node("ecology", ecology_desk)
builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", route, ["rules", "safety", "ecology"])
for name in ("rules", "safety", "ecology"):
    builder.add_edge(name, END)
desk = builder.compile()

for q in ("Do I need a permit for two hives?", "What is the sting risk next door?",
          "Will more hives make the park flowers bloom?"):  # third: the default route
    out = desk.invoke({"question": q}, config={"recursion_limit": 5})
    print(out["kind"], "->", out["reply"])

## The numbers, both directions

**Multi-agent is not a best practice. It is a purchase.**

- Anthropic's orchestrator-workers beat single-agent by 90.2% on research evals
- at roughly 15x the tokens of a plain chat session; spend explained 80% of the variance
- counter-example: Claude Code keeps one main loop; subagents return only summaries
- every layer multiplies debugging cost: add agents when a measurement pays for them

## Parallel tool calls

**Parallel tool calls are model behavior, not a request flag.**

- one AI reply may carry several `tool_calls`; nobody asked it to
- the loop's duty: iterate every entry, answer every id
- provider flags do not port: same code, different `MODEL_STRONG`, different fan-out
- `ToolNode` already iterates; your session-1 loop had to learn it

In [ ]:
from langchain_core.messages import AIMessage
from langchain_core.tools import tool
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode


@tool
def hive_density(district: str) -> str:
    """Registered hives per square km in a city district."""
    return {"old town": "12 per km2", "docks": "3 per km2"}.get(district, "no data")


@tool
def bloom_calendar(month: str) -> str:
    """What is flowering in the city parks in a given month."""
    return {"may": "chestnut, hawthorn", "july": "lime, clover"}.get(month, "no data")


def impatient_model(state: MessagesState) -> dict:
    # a fake model node: two tool calls in one reply
    return {"messages": [AIMessage(content="", tool_calls=[
        {"name": "hive_density", "args": {"district": "old town"}, "id": "call-a"},
        {"name": "bloom_calendar", "args": {"month": "july"}, "id": "call-b"},
    ])]}


builder = StateGraph(MessagesState)
builder.add_node("model", impatient_model)
builder.add_node("tools", ToolNode([hive_density, bloom_calendar]))
builder.add_edge(START, "model")
builder.add_edge("model", "tools")
builder.add_edge("tools", END)

out = builder.compile().invoke({"messages": []}, config={"recursion_limit": 5})
for message in out["messages"]:
    if message.type == "tool":  # one ToolMessage per entry, each id answered
        print(message.tool_call_id, "->", message.content)

## The blackboard

**Technique 22: one shared store, namespaces for private and collective scopes.**

- state flows along edges; a store sits beside the whole graph
- namespace tuples scope it: one corner per agent, one for the team
- session 5's `InMemoryStore` through the other door: `compile(store=...)`, the node signature takes `store`

In [ ]:
from langgraph.store.memory import InMemoryStore

BOARD = ("hive_survey", "shared")  # a namespace tuple: the team's corner


def field_scout(state: Survey, *, store) -> dict:
    store.put(BOARD, "old-town", {"fact": "12 hives, roof waiting list"})
    store.put(BOARD, "docks", {"fact": "3 hives, wind exposure flagged"})
    return {"notes": ["scout: posted 2 facts"]}


def board_reader(state: Survey, *, store) -> dict:
    facts = [item.value["fact"] for item in store.search(BOARD)]
    return {"notes": [f"reader: found {len(facts)} facts on the board"]}


builder = StateGraph(Survey)
builder.add_node("scout", field_scout)
builder.add_node("reader", board_reader)
builder.add_edge(START, "scout")
builder.add_edge("scout", "reader")
builder.add_edge("reader", END)
board_graph = builder.compile(store=InMemoryStore())  # the store rides beside the state

out = board_graph.invoke({"topic": "hive survey", "notes": []},
                         config={"recursion_limit": 5})
print(out["notes"])

## The research agent

**Everything above, assembled: one topic in, one report out.**

- a model drafts analyst personas as structured output
- a human edits the team through `interrupt()`
- `Send` fans out one interview subgraph per analyst
- sections merge through a reducer; one node writes the report

In [ ]:
from pydantic import BaseModel, Field


class Analyst(BaseModel):
    name: str = Field(description="Persona name")
    role: str = Field(description="One line: whose interests this analyst represents")


class Perspectives(BaseModel):
    analysts: list[Analyst] = Field(description="The full team of analyst personas")


print(list(Analyst.model_fields))

In [ ]:
# creation order is fixed: the offline harness hands out scripts by it
analyst_model = chat_model("strong")   # drafts the team, structured
question_model = chat_model("cheap")   # the interviewer in every branch
answer_model = chat_model("cheap")     # the expert voice, cites documents
writer_model = chat_model("cheap")     # one section per interview
report_model = chat_model("strong")    # merges sections into the report
solo_model = chat_model("strong")      # the single-agent baseline

print(type(analyst_model).__name__)

**`with_structured_output`: the reply arrives parsed, as your Pydantic object.**

- the schema is the payload contract: downstream code reads fields, not prose
- a list a human can count, reorder, and edit
- under the hood: a tool call named after the schema

In [ ]:
TOPIC = "Should our city encourage rooftop beehives?"

structured_analyst = analyst_model.with_structured_output(Perspectives)


def analyst_prompt(topic: str, feedback: str) -> str:
    return (f"Assemble three to five analyst personas for a report on: {topic}. "
            f"Editor feedback to honour: {feedback or 'none yet'}.")


team = structured_analyst.invoke(analyst_prompt(TOPIC, ""))
print(type(team).__name__)  # a parsed object, not text to regex
for analyst in team.analysts:
    print("-", analyst.name, "|", analyst.role)

## The corpus and the search stub

**Six documents in a dict. The interviews search these, offline.**

- deterministic and keyless: you debug the orchestration, not an API
- a miss returns advice, not an exception: session 1's `find_book` rule
- swap in real search later; the graph will not notice

In [ ]:
CORPUS = {
    "Doc 1": "City survey 2025: 12 registered rooftop hives in the old town, 3 near the docks.",
    "Doc 2": "Regulation draft: at most two hives per roof, each registered with the city council.",
    "Doc 3": "Yield study: an urban hive averages 18 kg of honey per season, above the rural mean.",
    "Doc 4": "Pollinator note: bees forage within two miles; balcony flowers set measurably more seed.",
    "Doc 5": "Complaint log: two sting reports in five years, both during unlicensed hive moves.",
    "Doc 6": "Insurance memo: standard building policies cover registered hives without extra clauses.",
}


def stub_search(query: str) -> str:
    """Keyword lookup over CORPUS: an offline stand-in for web search."""
    words = [w for w in query.lower().split() if len(w) > 3]
    hits = [f"[{doc_id}] {text}" for doc_id, text in CORPUS.items()
            if any(w in text.lower() for w in words)]
    if not hits:  # a miss the model can act on: session 1's find_book rule
        return "No documents matched. Try one concrete keyword: hives, permit, honey, sting."
    return "\n".join(hits[:2])  # two documents at most keeps the context small


print(stub_search("how many hives are registered"))
print()
print(stub_search("municipal tax revenue"))

## The interview subgraph

**Its own schema. Private transcript in, one section out.**

- `messages` and `docs` never surface in the parent state
- the router caps turns by counting AI messages; it never parses content
- two question turns per analyst: enough to watch the loop close

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph.message import add_messages


class InterviewState(TypedDict):
    analyst: dict
    messages: Annotated[list, add_messages]  # private transcript, never leaves
    docs: Annotated[list[str], operator.add]
    sections: Annotated[list[str], operator.add]  # the only key the parent sees


def ask_question(state: InterviewState) -> dict:
    prompt = (f"You are {state['analyst']['name']}, {state['analyst']['role']}. "
              "Ask the expert one short question about the topic.")
    reply = question_model.invoke([SystemMessage(prompt), *state["messages"]])
    return {"messages": [reply]}  # the reply object itself, never rebuilt


def search_docs(state: InterviewState) -> dict:
    return {"docs": [stub_search(state["messages"][-1].content)]}


def answer_question(state: InterviewState) -> dict:
    prompt = ("You are the expert. Answer the last question, cite sources "
              "as [Doc N].\n" + "\n".join(state["docs"]))
    reply = answer_model.invoke([SystemMessage(prompt), *state["messages"]])
    return {"messages": [reply]}


def more_questions(state: InterviewState) -> str:
    turns = sum(m.type == "ai" for m in state["messages"]) // 2  # count, never parse
    return "ask_question" if turns < 2 else "write_section"


def write_section(state: InterviewState) -> dict:
    prompt = (f"Write a report section from {state['analyst']['name']}'s interview. "
              "Keep the [Doc N] citations.\n" + "\n".join(state["docs"]))
    reply = writer_model.invoke([SystemMessage(prompt), *state["messages"]])
    return {"sections": [f"{state['analyst']['name']}: {reply.content}"]}


builder = StateGraph(InterviewState)
builder.add_node("ask_question", ask_question)
builder.add_node("search_docs", search_docs)
builder.add_node("answer_question", answer_question)
builder.add_node("write_section", write_section)
builder.add_edge(START, "ask_question")
builder.add_edge("ask_question", "search_docs")
builder.add_edge("search_docs", "answer_question")
builder.add_conditional_edges("answer_question", more_questions,
                              ["ask_question", "write_section"])
builder.add_edge("write_section", END)
interview = builder.compile()

print(interview.get_graph().draw_mermaid())

## The top graph

**Session 6's `interrupt()`, back for plan editing, not tool confirmation.**

- the pause gates a fan-out: bad team in, four bad interviews out
- the router returns a node name to loop, or a list of `Send` to launch
- a checkpointer is mandatory: session 6's rule, resume needs a parked state

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, interrupt


class ResearchState(TypedDict):
    topic: str
    feedback: str
    analysts: list  # replaced on regeneration: no reducer on purpose
    sections: Annotated[list[str], operator.add]  # merged from every branch
    report: str


def create_analysts(state: ResearchState) -> dict:
    team = structured_analyst.invoke(analyst_prompt(state["topic"], state["feedback"]))
    return {"analysts": [a.model_dump() for a in team.analysts]}


def human_feedback(state: ResearchState) -> dict:
    names = [a["name"] for a in state["analysts"]]
    decision = interrupt({"team": names, "hint": "type feedback, or approve"})
    return {"feedback": decision}  # the resume value is interrupt()'s return


def route_or_fan(state: ResearchState):
    if state["feedback"].strip().lower() != "approve":
        return "create_analysts"  # a node name: loop back with the note
    return [Send("interview",  # or a list of Send: one branch per analyst
                 {"analyst": analyst,
                  "messages": [HumanMessage(f"Interview focus: {state['topic']}")],
                  "docs": [], "sections": []})
            for analyst in state["analysts"]]


def write_report(state: ResearchState) -> dict:
    prompt = ("Merge these sections into one short report:\n\n"
              + "\n\n".join(state["sections"]))
    return {"report": report_model.invoke(prompt).content}


builder = StateGraph(ResearchState)
builder.add_node("create_analysts", create_analysts)
builder.add_node("human_feedback", human_feedback)
builder.add_node("interview", interview)  # the compiled subgraph is a node
builder.add_node("write_report", write_report)
builder.add_edge(START, "create_analysts")
builder.add_edge("create_analysts", "human_feedback")
builder.add_conditional_edges("human_feedback", route_or_fan,
                              ["create_analysts", "interview"])
builder.add_edge("interview", "write_report")
builder.add_edge("write_report", END)
research = builder.compile(checkpointer=InMemorySaver())  # resume needs a parked state

print(research.get_graph(xray=1).draw_mermaid())  # xray expands the subgraph

In [ ]:
config = {"configurable": {"thread_id": "report-1"}, "recursion_limit": 50}
# 50, not 8: every parallel branch spends supersteps of its own

paused = research.invoke({"topic": TOPIC, "feedback": "", "analysts": [],
                          "sections": [], "report": ""}, config=config)

print("paused with:", paused["__interrupt__"][0].value)
for analyst in paused["analysts"]:  # draft one: the team it proposed
    print("-", analyst["name"], "|", analyst["role"])

In [ ]:
revised = research.invoke(Command(resume="Add a public health expert to the team."),
                          config=config)

for analyst in revised["analysts"]:  # draft two: the requested expert appears
    print("-", analyst["name"], "|", analyst["role"])
print("paused again:", "__interrupt__" in revised)

In [ ]:
done = research.invoke(Command(resume="approve"), config=config)

print(len(done["sections"]), "sections, one per analyst")
print()
print(done["report"])

**One private interview per analyst ran in parallel. One reducer collected them.**

- each branch kept its own transcript; only `sections` crossed the wall
- the merge is task order, so the section order is reproducible
- `recursion_limit` grew to 50: branches spend supersteps; the default is 10007

## What the team costs

**Count what you can in code. Read true tokens from the trace.**

- call counts and character sizes are computable right here, deterministically
- the solo baseline reads the same corpus in one strong call
- the ratio below is your own 15x, before token truth arrives

In [ ]:
solo_prompt = ("Write a short report on: " + TOPIC + "\nSources:\n"
               + "\n".join(f"[{d}] {t}" for d, t in CORPUS.items()))
solo_report = solo_model.invoke(solo_prompt).content

team_size = len(done["analysts"])
team_calls = 2 + team_size * 5 + 1  # 2 drafts + 5 per interview + 1 report
# deliverables only: interview chatter is output too; the trace has the rest
team_chars = sum(len(s) for s in done["sections"]) + len(done["report"])

print(f"{'':10}{'model calls':>12}{'deliverable chars':>18}")
print(f"{'solo':10}{1:>12}{len(solo_report):>18}")
print(f"{'team':10}{team_calls:>12}{team_chars:>18}")
print("call ratio:", team_calls, "to 1 | true token counts: read the trace")

In [ ]:
from langfuse import get_client
from langfuse.langchain import CallbackHandler

client = get_client()  # reads LANGFUSE_HOST and both keys from the environment
print("server:", os.getenv("LANGFUSE_HOST"), "| up:", client.auth_check())

handler = CallbackHandler()  # each invoke is its own trace unless a span wraps them
traced_cfg = {"configurable": {"thread_id": "report-traced"},
              "recursion_limit": 50, "callbacks": [handler]}

# one parent span: the invoke and both resumes nest under it
with client.start_as_current_observation(name="research-run", as_type="span"):
    research.invoke({"topic": TOPIC, "feedback": "", "analysts": [],
                     "sections": [], "report": ""}, config=traced_cfg)
    research.invoke(Command(resume="Add a public health expert to the team."),
                    config=traced_cfg)
    traced = research.invoke(Command(resume="approve"), config=traced_cfg)
solo_model.invoke(solo_prompt, config={"callbacks": [handler]})  # the baseline: its own trace

client.flush()  # a notebook kernel never exits, so nothing sends without this
print(len(traced["sections"]), "sections in the traced run")

**Find the fan-out in the trace tree.**

- one tree for the run: the invoke and two resumes under `research-run`; the solo baseline is a trace of its own
- interview spans sit side by side, overlapping in time
- open a branch: its private transcript, its searches, its section
- token totals, team vs solo: your own version of the 15x

## Practice

**The demo is the practice today: rebuild it over your own corpus.**

1. six to eight stub documents in your project domain; the search stays offline
2. analyst generation with `with_structured_output`; your edit goes through `interrupt()`
3. interview subgraph with a private schema, `Send` fan-out, reducer merge
4. every `invoke` passes a `recursion_limit` sized for branches times turns

**Required artifact: `runs/session-09.md`, plus the trace that proves the parallelism.**

5. analysts before and after your edit, and the merged report
6. the solo-vs-team table, with token totals read from your traces
7. export the trace whose tree shows the parallel branches
8. stretch: real web search behind `stub_search`'s interface; the artifact must still reproduce without keys

## Declare your architecture

**Each student, out loud: one loop, subgraphs, or an orchestrator. One sentence why.**

- "one loop, because my tools are dependent" is a strong answer
- research-topic projects: today's pattern ports into the project directly
- the defense asks again, with your trace on the screen

## Today, in one card

**A team is shared state plus a merge policy; multi-agent is a purchase, priced in tokens.**

**You can now defend:**
- two nodes writing one key in a superstep is an error, not a race; a reducer is the merge policy that allows it
- `Send` fans out per data item with its own payload; a subgraph keeps private keys, and one reduced key is the whole contract
- orchestrator-workers beat a single agent on research evals at roughly 15x the tokens; add agents when a measurement pays for them

**In your repository:** `runs/session-09.md`, analysts before and after your edit, the merged report, the solo-vs-team table, the parallel trace.
**The trap of the day:** a `recursion_limit` sized for one loop kills a fan-out: every branch spends supersteps of its own.
**Ask yourself:** one reply carries several `tool_calls`: whose job is it to answer every id, and why does the fan-out change with `MODEL_STRONG`?
**Next time:** autoresearch: a human fixes the metric, and the agent must beat it.